In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, FloatType, DateType

bronze_sales_schema = StructType([
    StructField("InvoiceNo", StringType(), False),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", FloatType(), True),
    StructField("InvoiceDate", DateType(), True),
    StructField("UnitPrice", FloatType(), True),
    StructField("CustomerID", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("Discount", FloatType(), True),
    StructField("PaymentMethod", StringType(), True),
    StructField("ShippingCost", FloatType(), True),
    StructField("Category", StringType(), True),
    StructField("SalesChannel", StringType(), True),
    StructField("ReturnStatus", StringType(), True),
    StructField("ShipmentProvider", StringType(), True),
    StructField("WarehouseLocation", StringType(), True),
    StructField("OrderPriority", StringType(), True)
])

In [0]:
def createRawDf():
    raw_df = (
        spark.read.format("csv")
        .option("header", "true")
        .schema(bronze_sales_schema)
        .load("/Volumes/exp_catalog/bronze/vol_bronze")
    )
    return raw_df

createRawDf

In [0]:
raw_df.limit(10).display()

In [0]:
#bronze_sales_schema.fields[0].nullable = False

raw_df.printSchema()
raw_df.count()

In [0]:
bronze_sales_df = (
    raw_df
    .withColumn("_ingestedTs", F.current_timestamp())
    .withColumn("_ingestedDate", F.date_format(F.current_timestamp(), "yyyy-MM-dd"))
#    .withColumn("_inputFileName", F.input_file_name())
#    .withColumn("_inputFileSize", F.size(F.input_file_name()))
#    .withColumn("_inputFileBlockId", F.block_id())
#    .withColumn("_inputFileOffset", F.offset())
#    .withColumn("_inputFilePartitionId", F.partition_id())
#    .withColumn("_inputFilePartitionOffset", F.partition_offset())
#    .withColumn("_inputFilePartitionSize", F.partition_size())
#    .withColumn("_inputFilePartitionTotal", F.partition_total())
)

In [0]:
bronze_sales_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_sales")

In [0]:
spark.sql("select * from bronze_sales limit 10")
display(spark.table("bronze_sales").limit(10))

spark.sql("""
SELECT 
    COUNT(*) AS total_rows,
    COUNT(DISTINCT InvoiceNo) AS distinct_invoices,
    COUNT(DISTINCT CustomerID) AS distinct_customers
FROM bronze_sales
""").show()
